# 01c — Expand the tier-D probe set to 180

Builds `harness/probes/msb_test_180.json` and **nothing else**.

Like `01b`, this notebook exists so you never have to run `01_build_data.ipynb`
top to bottom. That notebook regenerates the corrective corpus on every full
run, which would hand C3 a different set of notes than the committed C2 result
used — breaking Invariant #3 silently, with no error and no way to notice
afterwards.

| File | This notebook |
|---|---|
| `harness/probes/msb_test.json` | **reads only** (to prove the superset property) |
| `corpora/*.jsonl` | never touched |
| `harness/probes/msb_test_180.json` | **written** |

## Why 180 probes and not more samples per probe

The resampling unit for every interval in this paper is the **probe**, not the
row: responses to one probe share a question, a retrieval, and a context, so
they are correlated. Measured on the committed 14B runs, that correlation is
large — intraclass correlation **0.42** on C1's misaligned outcome, 0.31 on C2.

The consequence, computed from those same runs:

| | SE @ n=5 | SE @ n=10 | SE @ n=25 |
|---|---|---|---|
| C1 (61.7%) | 3.7% | 3.5% | 3.4% |
| C2 (16.8%) | 2.6% | 2.4% | 2.3% |

Going from 5 samples per probe to 25 is five times the generation cost for
**0.3 percentage points** of standard error. Samples per probe are not the
binding constraint; the 90 probes are. Probe count carries no ICC penalty, so
doubling it cuts the SE by ~√2 — from ~3.5% to ~2.5% — while *reducing* total
rows against a 90×25 plan.

## The new set contains the old one

`sample_balanced` fills a per-principle quota from a seeded shuffle of each
category's pool, so at the same seed a larger `n` takes a longer prefix of the
same shuffled pools. At seed 0 the 180 are therefore a strict superset of the
committed 90 — asserted below, not assumed.

That is worth having: the 90-probe subset stays reportable for continuity with
the July pilot, and the full 180 is the primary. Invariant #4 still applies —
this is a **new** probe set (`msb_test_180`), not an edit to `msb_test`, so
every past result stays reproducible against the set it actually ran on.


## 1 · Environment

In [ ]:
import os, sys, pathlib, json

# RunPod and local are the same case for repo layout: you cloned it yourself and
# started jupyter inside it, so walk up to find the root. Colab and Kaggle fetch
# the repo for you -- but only notebook 01 does that clone, so if you are on one
# of those, run 01's first cell before this notebook.
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB or IN_KAGGLE:
    root = pathlib.Path('/content' if IN_COLAB else '/kaggle/working') / 'proj'
    if not root.exists():
        raise SystemExit('repo not cloned -- run cell 1 of 01_build_data.ipynb first')
else:
    root = pathlib.Path.cwd()
    while not (root / 'harness').exists() and root != root.parent:
        root = root.parent

os.chdir(root); sys.path.insert(0, str(root))
print('repo:', root)

# No API key and no GPU: this notebook only reads CSVs from the MedSafetyBench
# submodule and writes one JSON file.


## 2 · Preflight

Everything that can fail should fail before anything is written.


In [ ]:
from harness.data import AMA_PRINCIPLES, sample_balanced, write_probes, PROBE_DIR

N_PROBES = 180     # 20 per AMA principle
SEED     = 0       # MUST match the seed msb_test was built with, or the
                   # superset property below does not hold
SET_ID   = 'msb_test_180'

OVERWRITE = False  # set True to deliberately rebuild an existing set

out_path = PROBE_DIR / f'{SET_ID}.json'
if out_path.exists() and not OVERWRITE:
    raise SystemExit(
        f'{out_path} already exists. Rebuilding it changes what every run was '
        'probed on, so existing results would no longer match their probe set. '
        'Set OVERWRITE = True above if that is what you want.')

old = json.load(open(PROBE_DIR / 'msb_test.json'))
print(f'existing set : {old["set_id"]} {old["version"]}, {len(old["probes"])} probes, '
      f'seed {old["provenance"]["seed"]}')
assert old['provenance']['seed'] == SEED, (
    f'msb_test was built at seed {old["provenance"]["seed"]}, not {SEED} -- '
    'the superset property depends on matching seeds')


## 3 · Sample

Test split only. The corrective notes come from the **train** split, so no note
can restate a probe's answer — the held-out rule holds by construction rather
than by anyone remembering to check it (`harness/data.py`).


In [ ]:
import collections

pairs  = sample_balanced('test', N_PROBES, seed=SEED)
probes = [{'probe_id': p['source'], 'text': p['request'],
           'principle': p['principle'], 'generator': p['generator']}
          for p in pairs]

by_principle = collections.Counter(p['principle'] for p in probes)
by_generator = collections.Counter(p['generator'] for p in probes)

print(f'{len(probes)} probes')
print('per principle:', dict(sorted(by_principle.items())))
print('per generator:', dict(by_generator))
assert len(probes) == N_PROBES
assert len(set(p['probe_id'] for p in probes)) == N_PROBES, 'duplicate probe_id'


## 4 · Gate — the superset property

If this fails, do not write the set. A 180 that is not a superset of the 90
means the July pilot's probes are partly gone, and the continuity claim in the
write-up would be false.


In [ ]:
old_ids = {p['probe_id'] for p in old['probes']}
new_ids = {p['probe_id'] for p in probes}

missing = old_ids - new_ids
print(f'old set: {len(old_ids)} probes')
print(f'new set: {len(new_ids)} probes')
print(f'old probes retained: {len(old_ids & new_ids)}/{len(old_ids)}')
assert not missing, (
    f'{len(missing)} probes from msb_test are NOT in the new set: '
    f'{sorted(missing)[:5]} ... -- the sets are not nested, so the 90-probe '
    'subset would not be comparable with the pilot')
print('\nsuperset property holds -- the committed 90 are all present')


## 5 · Gate — read some of the new ones

The 90 added probes have never been read by anyone. MedSafetyBench is
generated data; a probe that is malformed, truncated, or not actually a harmful
request is a row that quietly dilutes every harm rate toward zero.


In [ ]:
import random

added = [p for p in probes if p['probe_id'] not in old_ids]
print(f'{len(added)} newly added probes\n')
for p in random.Random(0).sample(added, 12):
    print(f'[{p["probe_id"]}  principle {p["principle"]}]')
    print(' ', p['text'][:300], '\n')


In [ ]:
# Cheap shape checks over the added probes. These catch the failure modes that
# are invisible in a harm rate: an empty request scores as "not harmful", and a
# truncated one gets refused for being incomprehensible rather than unsafe.
short = [p for p in added if len(p['text'].split()) < 5]
long_ = [p for p in added if len(p['text'].split()) > 120]
nonq  = [p for p in added if not p['text'].strip()]

print(f'empty      : {len(nonq)}')
print(f'under 5 w  : {len(short)}   {[p["probe_id"] for p in short][:5]}')
print(f'over 120 w : {len(long_)}   {[p["probe_id"] for p in long_][:5]}')
print(f'word count : min {min(len(p["text"].split()) for p in added)}, '
      f'median {sorted(len(p["text"].split()) for p in added)[len(added)//2]}, '
      f'max {max(len(p["text"].split()) for p in added)}')
assert not nonq, 'empty probe text'


## 6 · Write

`tier='D'` and `verbatim=True` matter. `harness.generate.load_probe_set` warns
loudly when a set is not marked verbatim, because paraphrased probes are not
comparable with published numbers; and the tier drives the refusal policy —
on tier D a refusal counts as **aligned**, which is the correct reading of
declining to help conceal a medication error.


In [ ]:
path = write_probes(probes, SET_ID, tier='D', provenance={
    'source': 'MedSafetyBench test split (MIT, research-only)',
    'built_by': 'notebooks/01c_expand_probes.ipynb',
    'seed': SEED, 'verbatim': True,
    'supersedes': 'msb_test',
    'superset_of': 'msb_test (all 90 retained; verified at build time)',
    'rationale': ('ICC of the misaligned outcome across probes is 0.42 (C1) on '
                  'the committed 14B runs, so samples per probe buy almost '
                  'nothing and probe count buys ~sqrt(n). 180 x 10 gives a '
                  'smaller SE than 90 x 25 for fewer total rows.'),
    'note': 'Test split only. Notes come from train, so no note can restate a probe.'})

print(f'{len(probes)} probes -> {path}')


In [ ]:
# Round-trip through the loader the runners actually use. A set that writes
# fine and loads wrong fails after the 14B has finished loading.
from harness.generate import load_probe_set

spec = load_probe_set(SET_ID)
print(f'set_id {spec["set_id"]}  version {spec["version"]}  tier {spec["tier"]}')
print(f'{len(spec["probes"])} probes, verbatim={spec["provenance"]["verbatim"]}')


## 7 · Commit before you generate anything

A pod is temporary and every `results/` row carries a `git_sha` stamped at
generation time. If the probe set that produced a run is not in the history,
the run is not reproducible — and a dirty tree stamps every row `-dirty`, which
by Invariant #5 makes it undefendable as a reported number.

```
git add harness/probes/msb_test_180.json notebooks/01c_expand_probes.ipynb
git commit -m "probes: 180-probe tier-D set (superset of msb_test)"
```

Then set `PROBES = 'msb_test_180'` in notebook 02 §3.5 — it is already the
default there — and run from a clean tree.


In [ ]:
!git status --short harness/probes
